# Guildmaster-AI Playground

Comprehensive interactive notebook covering all framework features.

Make sure your `.env` file has `OPENROUTER_API_KEY` set.

> **New here?** Start with the focused notebooks instead:
> - [`01_quickstart.ipynb`](01_quickstart.ipynb) — 5-line minimal setup
> - [`02_custom_adventurer.ipynb`](02_custom_adventurer.ipynb) — Subclassing BaseAdventurer
> - [`03_weapons_and_armor.ipynb`](03_weapons_and_armor.ipynb) — Tools and guardrails
> - [`04_guard.ipynb`](04_guard.ipynb) — LLM-as-judge verification
> - [`05_guild_info.ipynb`](05_guild_info.ipynb) — Inspecting guild state and quest history

In [ ]:
import os

from dotenv import load_dotenv

# Load .env from project root (works whether kernel runs from repo root or examples/)
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env")) or load_dotenv(".env")

# Ensure we're working from the project root for file paths
if os.path.basename(os.getcwd()) == "examples":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()
print(f"Project root: {PROJECT_ROOT}")
print(f"API key loaded: {'yes' if os.getenv('OPENROUTER_API_KEY') else 'no'}")

## 1. Basic Quest — Simple Q&A

The simplest usage: create a Guild with an OpenRouter LLM and a GeneralAdventurer,
then post a quest.

In [ ]:
from guildmaster_ai import GuildBuilder
from guildmaster_ai.adventurers.general_adventurer import GeneralAdventurer

guild = (
    GuildBuilder().with_llm_provider("openrouter").register_adventurer(GeneralAdventurer).build()
)

print("Guild created!")
print(f"Roster: {[p.name or p.id for p in guild.roster]}")

In [ ]:
result = await guild.run_quest(
    "What are the 3 key differences between Python and Rust? Be concise."
)

print(f"Success: {result.success}")
print("---")
print(result.summary)

## 2. Using the LLM Directly

You can use the LLM layer directly without the full quest lifecycle.
`guild_complete()` is a simple helper for one-shot completions.

In [ ]:
from guildmaster_ai.llm import create_chat_model, guild_complete

llm = create_chat_model("openrouter")

answer = await guild_complete(
    llm,
    system="You are a helpful assistant that responds in haiku format.",
    user="Tell me about async programming.",
)
print(answer)

## 3. Equipping Weapons (Tools)

Adventurers can use weapons (tools) during quests. The LLM decides when to call them.
Here we equip the `FileReadWeapon` so the adventurer can read files.

In [ ]:
from guildmaster_ai.weapons.file_read import FileReadWeapon

adventurer = GeneralAdventurer(name="FileReader")
adventurer.equip_weapon(FileReadWeapon())

guild_with_tools = (
    GuildBuilder().with_llm_provider("openrouter").register_adventurer(adventurer).build()
)

# Use absolute path so it works regardless of kernel working directory
toml_path = os.path.join(PROJECT_ROOT, "pyproject.toml")

result = await guild_with_tools.run_quest(
    f"Read the file '{toml_path}' and tell me what Python version is required "
    "and list all the core dependencies. Be concise."
)

print(f"Success: {result.success}")
print("---")
print(result.summary)

## 4. Armor (Guardrails)

Armor provides pre/post processing guardrails. The `ContentFilterArmor`
blocks messages matching forbidden patterns.

In [ ]:
from guildmaster_ai.armor.content_filter import ContentFilterArmor

guarded_adventurer = GeneralAdventurer(name="GuardedAdventurer")
guarded_adventurer.wear_armor(ContentFilterArmor(blocked_patterns=["password", "secret"]))

guild_guarded = (
    GuildBuilder().with_llm_provider("openrouter").register_adventurer(guarded_adventurer).build()
)

# This quest should work fine
result = await guild_guarded.run_quest("What is 2 + 2? Answer in one word.")
print(f"Clean quest - Success: {result.success}")
print(f"Answer: {result.summary}")

print("\n---\n")

# This quest should be blocked by the content filter (returns failed result)
result = await guild_guarded.run_quest("What is my password?")
print(f"Blocked! Success: {result.success}")
print(f"Reason: {result.summary}")

## 5. Guard (LLM-as-Judge)

Enable the Guard to have an LLM evaluate quest results for quality and safety.

In [ ]:
guild_with_guard = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(GeneralAdventurer)
    .with_guard()
    .build()
)

result = await guild_with_guard.run_quest(
    "Write a short Python function that checks if a number is prime."
)

print(f"Success: {result.success}")
print("---")
print(result.summary)

## 6. Custom Adventurer

Create your own adventurer by subclassing `BaseAdventurer`.

In [ ]:
from guildmaster_ai.adventurers.base_adventurer import BaseAdventurer
from guildmaster_ai.core.messages import QuestResult
from guildmaster_ai.core.quest import Quest


class PirateAdventurer(BaseAdventurer):
    """An adventurer that speaks like a pirate!"""

    @property
    def system_prompt(self) -> str:
        return (
            "You are a pirate adventurer! You speak in pirate dialect "
            "(arr, matey, ye, etc.) but still provide accurate and helpful answers. "
            "Keep responses concise \u2014 2-3 sentences max."
        )

    async def execute(self, quest: Quest) -> QuestResult:
        response = await self._llm_complete(quest.description)
        return QuestResult(
            sender=self.name or self.id,
            quest_id=quest.id,
            success=True,
            summary=response,
        )


guild_pirate = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(PirateAdventurer(name="Captain Hook"))
    .build()
)

result = await guild_pirate.run_quest("Explain what a REST API is.")
print(result.summary)

## 7. Pass Any LangChain Model

You can pass any LangChain `BaseChatModel` directly to the builder.
This works with any LangChain-compatible model.

In [ ]:
from guildmaster_ai.llm.openrouter import ChatOpenRouter

# Create a custom-configured LLM (different model, lower temperature)
custom_llm = ChatOpenRouter(
    model="meta-llama/llama-4-scout",
    temperature=0.3,
    max_tokens=256,
)

guild_custom = (
    GuildBuilder()
    .with_llm_provider(custom_llm)  # pass the LangChain model directly
    .register_adventurer(GeneralAdventurer)
    .build()
)

result = await guild_custom.run_quest("What is the capital of France? Answer in one word.")
print(result.summary)

## 8. Guild Info & Quest Tracking

Inspect the guild state, look up quests by UUID, and see the full lifecycle history.

In [ ]:
# Build a guild with a well-equipped adventurer
equipped = GeneralAdventurer(name="Scout")
equipped.equip_weapon(FileReadWeapon())

guild_inspect = GuildBuilder().with_llm_provider("openrouter").register_adventurer(equipped).build()

# Before any quests
print("=== Guild Info ===")
print(repr(guild_inspect))
print(f"Guild ID: {guild_inspect.info.id}")
print()

for p in guild_inspect.info.adventurers:
    print(f"Adventurer: {p.name}")
    print(f"  Talents: {p.talents}")
    print(f"  Weapons: {p.weapons}")
    print(f"  Armor:   {p.armor}")

# Run two quests
r1 = await guild_inspect.run_quest("What is 1 + 1? Answer with just the number.")
r2 = await guild_inspect.run_quest("Capital of Japan? One word.")

print("\n=== After Quests ===")
info = guild_inspect.info
print(f"Total: {info.total_quests}  Completed: {info.completed}  Failed: {info.failed}")

# List all quests
print("\n=== All Quests ===")
for q in guild_inspect.quests:
    print(f"  [{q.id[:8]}...] {q.title} — {q.status.value}")

# Look up a quest by UUID and inspect its history
quest_id = guild_inspect.quests[0].id
q = guild_inspect.get_quest(quest_id)
r = guild_inspect.get_result(quest_id)

print(f"\n=== Quest Detail: {q.title} ===")
print(f"ID:     {q.id}")
print(f"Status: {q.status.value}")
print(f"Rank:   {q.rank.name}")
print(f"Result: {r.summary if r else 'N/A'}")
print("History:")
for h in q.history:
    fr = h.payload.get("from", "")
    to = h.payload.get("to", "")
    print(f"  {fr} -> {to}  (by {h.actor})")